In [0]:
dbutils.library.restartPython()

In [0]:
import os
print(os.listdir("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src/energy_lakehouse"))

In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

In [0]:
import uuid
from datetime import UTC, datetime

import httpx

from energy_lakehouse.bronze import bronze_rows, chunks_to_fetch, load_window
from energy_lakehouse.http_client import NotFoundError
from energy_lakehouse.smard import (
    BASE_URL,
    PROJECT_INPUT_IDS,
    fetch_chunk,
    fetch_index,
    parse_chunk,
)

TABLE = "workspace.bronze.smard_raw"
DAYS_BACK = 90

dbutils.widgets.text("backfill_start", "", "Backfill start (YYYY-MM-DD)")
dbutils.widgets.text("backfill_end", "", "Backfill end (YYYY-MM-DD)")

load_id = str(uuid.uuid4())
fetched_at = datetime.now(tz=UTC)
window = load_window(
    dbutils.widgets.get("backfill_start"),
    dbutils.widgets.get("backfill_end"),
    fetched_at,
    DAYS_BACK,
)
print(load_id, window)

In [0]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

BRONZE_SCHEMA = StructType(
    [
        StructField("filter_id", IntegerType(), nullable=False),
        StructField("timestamp_utc", TimestampType(), nullable=False),
        StructField("value", DoubleType(), nullable=True),
        StructField("source_chunk", LongType(), nullable=False),
        StructField("load_id", StringType(), nullable=False),
        StructField("fetched_at", TimestampType(), nullable=False),
    ]
)

if not spark.catalog.tableExists(TABLE):
    spark.createDataFrame([], BRONZE_SCHEMA).write.saveAsTable(TABLE)

In [0]:
from pyspark.sql import functions as F

marks: dict[int, int] = {}
if spark.catalog.tableExists(TABLE):
    found = (
        spark.table(TABLE)
        .groupBy("filter_id")
        .agg(F.max("source_chunk").alias("hwm"))
        .collect()
    )
    marks = {row["filter_id"]: row["hwm"] for row in found}

print(marks)

In [0]:
with httpx.Client(base_url=BASE_URL, timeout=30) as client:
    for filter_id in PROJECT_INPUT_IDS:
        index = fetch_index(client, filter_id)
        mark = None if window.is_backfill else marks.get(filter_id)
        chunks = chunks_to_fetch(index, mark, window.start, window.end)

        rows = []
        for chunk in chunks:
            try:
                body = fetch_chunk(client, filter_id, chunk)
            except NotFoundError:
                print(f"  {filter_id}: week {chunk} missing, skipped")
                continue
            rows.extend(bronze_rows(parse_chunk(body, filter_id), chunk, load_id, fetched_at))

        if not rows:
            print(f"{filter_id}: nothing new")
            continue

        df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
        df.createOrReplaceTempView("incoming")
        spark.sql(f"""
            MERGE INTO {TABLE} AS target
            USING incoming AS source
               ON target.filter_id = source.filter_id
              AND target.timestamp_utc = source.timestamp_utc
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"{filter_id}: {len(chunks)} weeks, {len(rows)} rows written")

In [0]:
%sql
SELECT filter_id,
       count(*) AS n_rows,
       count(value) AS n_values,
       min(timestamp_utc) AS first_hour,
       max(timestamp_utc) AS last_hour,
       count(DISTINCT load_id) AS n_loads
FROM workspace.bronze.smard_raw
GROUP BY filter_id
ORDER BY filter_id

In [0]:
%sql
SELECT count(*) AS n_rows, count(DISTINCT load_id) AS n_loads
FROM workspace.bronze.smard_raw

In [0]:
%sql
SELECT count(*) AS count
FROM workspace.bronze.smard_raw

In [0]:
# %sql
# --- fixing code no need to run multiple times
# CREATE OR REPLACE TABLE workspace.bronze.smard_raw_dedup AS
# SELECT filter_id, timestamp_utc, value, source_chunk, load_id, fetched_at
# FROM (
#   SELECT *,
#          row_number() OVER (
#            PARTITION BY filter_id, timestamp_utc
#            ORDER BY fetched_at DESC, load_id DESC
#          ) AS rn
#   FROM workspace.bronze.smard_raw
# )
# WHERE rn = 1

In [0]:
# %sql
# SELECT
#   (SELECT count(*) FROM workspace.bronze.smard_raw)       AS before_rows,
#   (SELECT count(*) FROM workspace.bronze.smard_raw_dedup) AS after_rows

In [0]:
# %sql
# DROP TABLE workspace.bronze.smard_raw;
# ALTER TABLE workspace.bronze.smard_raw_dedup RENAME TO workspace.bronze.smard_raw

In [0]:
# %sql
# -- old code below
# SELECT filter_id,
#        count(*)      AS n_rows,
#        count(value)  AS n_values,
#        min(timestamp_utc) AS first_hour,
#        max(timestamp_utc) AS last_hour,
#        count(DISTINCT load_id) AS n_loads
# FROM workspace.bronze.smard_raw
# GROUP BY filter_id
# ORDER BY filter_id

In [0]:
# %sql
# SELECT count(*) AS n_rows,
#        min(timestamp_utc) AS first_hour,
#        max(timestamp_utc) AS last_hour,
#        count(DISTINCT load_id) AS n_loads
# FROM workspace.bronze.smard_raw

In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.bronze.smard_raw